In [1]:
import pandas as pd
import numpy as np

In [2]:
x1 = np.arange(1, 101)
x2 = np.arange(10, 1010, 10)
y = np.array([1] * 10 + [0] * 90)

df = pd.DataFrame({
    'x1': x1,
    'x2': x2,
    'y': y
})

In [3]:
print(df.head())
print(df.tail())

   x1  x2  y
0   1  10  1
1   2  20  1
2   3  30  1
3   4  40  1
4   5  50  1
     x1    x2  y
95   96   960  0
96   97   970  0
97   98   980  0
98   99   990  0
99  100  1000  0


In [4]:
def manual_stratified_kfold(df, target_col, k=5, random_state=None):
    if random_state is not None:
        np.random.seed(random_state)
        
    # 1. Group indices by class
    classes = df[target_col].unique()
    class_indices = {c: df[df[target_col] == c].index.values for c in classes}
    
    # 2. Shuffle indices within each class to ensure randomness
    for c in classes:
        np.random.shuffle(class_indices[c])
        
    # 3. Split each class's indices into 'k' roughly equal chunks
    folds = [[] for _ in range(k)]
    for c in classes:
        # np.array_split safely handles arrays that aren't perfectly divisible by k
        splits = np.array_split(class_indices[c], k)
        for i in range(k):
            folds[i].extend(splits[i])
            
    # 4. Generate the train and test indices for each fold
    for i in range(k):
        test_idx = np.array(folds[i])
        
        # Train indices are all folds except the current test fold (i)
        train_idx = np.concatenate([folds[j] for j in range(k) if j != i])
        
        # Optional: Shuffle the final indices so classes aren't grouped together sequentially
        np.random.shuffle(train_idx)
        np.random.shuffle(test_idx)
        
        yield train_idx, test_idx

# --- Testing the implementation ---
k = 5
fold_number = 1

for train_idx, test_idx in manual_stratified_kfold(df, 'y', k=k, random_state=42):
    train_df = df.loc[train_idx]
    test_df = df.loc[test_idx]
    
    print(f"--- Fold {fold_number} ---")
    print(f"Train size: {len(train_df)} | Test size: {len(test_df)}")
    
    # Check the actual counts of 0s and 1s to verify stratification
    train_counts = train_df['y'].value_counts().to_dict()
    test_counts = test_df['y'].value_counts().to_dict()
    
    print(f"Train class counts: {train_counts}")
    print(f"Test class counts:  {test_counts}\n")
    
    fold_number += 1

--- Fold 1 ---
Train size: 80 | Test size: 20
Train class counts: {0: 72, 1: 8}
Test class counts:  {0: 18, 1: 2}

--- Fold 2 ---
Train size: 80 | Test size: 20
Train class counts: {0: 72, 1: 8}
Test class counts:  {0: 18, 1: 2}

--- Fold 3 ---
Train size: 80 | Test size: 20
Train class counts: {0: 72, 1: 8}
Test class counts:  {0: 18, 1: 2}

--- Fold 4 ---
Train size: 80 | Test size: 20
Train class counts: {0: 72, 1: 8}
Test class counts:  {0: 18, 1: 2}

--- Fold 5 ---
Train size: 80 | Test size: 20
Train class counts: {0: 72, 1: 8}
Test class counts:  {0: 18, 1: 2}

